# PRG recording: load data and identify cell types

Use the `neuro_tf221` kernel. Choose an HDF5 recording below. The file contains a single continuous trajectory after the discarded warm-up; arrays have shape **(time, core neurons)**. Spikes are binary uint8; voltages are float16 **normalized model voltages**, not mV (reset/rest = 0, threshold = 1).

Neuron columns use canonical network order. `neuron_ids` indexes this model's canonical neuron order; `bmtk_ids` identifies the corresponding neurons in the original SONATA data. Cell types are reconstructed from those IDs and the source network files, without loading the simulated network or a checkpoint.

## Reference checkpoint

The trained 203,816-neuron reference is `ckpt-2` from TalaIA run `b_u05m`, stored in
[`PRG/checkpoints/crowd_osi_ckpt/`](checkpoints/README.md). The original training
configuration and source checksums are preserved there. Its training used soft
reset; PRG intentionally records its learned weights with **hard reset** and
float32 inference.

From the repository root on Nuredduna, submit a long reference recording:
```bash
bash PRG/long_gray_simulation.sh --restore_from PRG/checkpoints/crowd_osi_ckpt
```

The notebook reads the resulting **HDF5 recording**, not the checkpoint weights.
Select a long-run HDF5 file below once one has been generated. The
metadata cell prints the checkpoint actually used; an untrained recording has
an empty checkpoint attribute. Changing the checkpoint path in a launch command
does not change an existing recording.


In [2]:
from pathlib import Path
import sys
from time import perf_counter

import h5py
import numpy as np
import pandas as pd

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "v1_model_utils").is_dir()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [3]:
# Replace with the generated HDF5 path from a long reference run when available.
recording_path = repo_root / "PRG/results/prg_20261028_103858/simulation_data.hdf5"

start_ms = 0
window_ms = 60000
max_neurons = 66408  # Bound memory use; the cell-type vector below covers every column.
load_workers = 4  # Parallel gzip decoding; use 1 for the standard HDF5 reader.


In [4]:
with h5py.File(recording_path, "r") as recording:
    metadata = dict(recording.attrs)
    neuron_ids = recording["neuron_ids"][:]
    bmtk_ids = recording["bmtk_ids"][:]
    completed_steps = int(metadata["completed_steps"])
    dt_ms = float(metadata["dt_ms"])
    dataset_info = {
        name: {"shape": recording[name].shape, "dtype": str(recording[name].dtype)}
        for name in ("spikes", "voltages")
    }

recorded_checkpoint = metadata.get("checkpoint", "")
print("Recorded checkpoint:", recorded_checkpoint or "None (--evaluate_untrained)")
print("Reference directory:", repo_root / "PRG/checkpoints/crowd_osi_ckpt")
print(metadata)
print(dataset_info)
print(f"Available recording: {completed_steps * dt_ms / 1000:g} s after warm-up")


Recorded checkpoint: PRG/checkpoints/crowd_osi_ckpt/ckpt-2
Reference directory: /home/jgalvan/Desktop/Neurocoding/V1_GLIF_model/PRG/checkpoints/crowd_osi_ckpt
{'checkpoint': 'PRG/checkpoints/crowd_osi_ckpt/ckpt-2', 'completed_steps': np.int64(60000), 'data_dir': '/home/jgalvan/Desktop/Neurocoding/V1_GLIF_model/GLIF_network_nll_full', 'dt_ms': np.int64(1), 'duration_ms': np.int64(60000), 'flags_json': '{"?": false, "acceleration": "auto", "alsologtostderr": false, "bmtk_compat_lgn": true, "caching": true, "ckpt_dir": "", "compress_model_cache": false, "connected_selection": true, "core_only": false, "cue_duration": 40, "dampening_factor": 0.05, "data_dir": "GLIF_network_nll_full", "delta_threshold": 0.5, "dtype": "float32", "duration_ms": 60000, "evaluate_untrained": false, "gauss_std": 0.28, "hard_reset": true, "hbm_oom_exit": true, "help": false, "helpfull": false, "helpshort": false, "helpxml": false, "input_weight_scale": 1.0, "integration_scheme": "exact", "lgn_backend": "grating",

## Load a bounded window

Only samples below `completed_steps` are valid if a session stopped early. The first stored sample follows one model step after warm-up; times below are measured from the end of warm-up. Read larger windows explicitly when needed, rather than loading the complete voltage dataset.

The reader decompresses gzip chunks on `load_workers` CPU threads, with bounded
temporary memory and no extra files. Other HDF5 layouts use the standard reader.
The full 60-second recording still needs about 11.1 GiB of RAM for both arrays.


In [5]:
from PRG.loading import read_window

if start_ms < 0 or window_ms <= 0 or max_neurons <= 0:
    raise ValueError("Use start_ms >= 0, window_ms > 0 and max_neurons > 0.")
start_step = min(int(start_ms / dt_ms), completed_steps)
stop_step = min(start_step + max(1, int(window_ms / dt_ms)), completed_steps)
n_columns = min(max_neurons, len(neuron_ids))

load_start = perf_counter()
with h5py.File(recording_path, "r") as recording:
    spikes = read_window(recording["spikes"], start_step, stop_step, n_columns, load_workers)
    voltages = read_window(recording["voltages"], start_step, stop_step, n_columns, load_workers)

time_ms = (np.arange(start_step, stop_step) + 1) * dt_ms
print(f"Loaded spikes {spikes.shape}, voltages {voltages.shape} in {perf_counter() - load_start:.2f} s")


Loaded spikes (60000, 66408), voltages (60000, 66408) in 20.77 s


In [6]:
np.min(voltages), np.max(voltages), np.mean(voltages), np.std(voltages)

/home/jgalvan/.conda/envs/neuro_tf221/lib/python3.12/site-packages/numpy/_core/_methods.py:168: RuntimeWarning: overflow encountered in reduce
  arrmean = umr_sum(arr, axis, dtype, keepdims=True, where=where)


(np.float16(-2.86), np.float16(2.336), np.float16(0.5635), np.float16(inf))

## Recover the aligned cell-type vector

Use `bmtk_ids` for lookup in the original SONATA files. Canonical `neuron_ids` alone are not SONATA row IDs. The existing population-name utilities accept a minimal neuron-ID mapping, so no network construction is required. Keep the default L5 excitatory subtype distinction.


In [7]:
from v1_model_utils.other_v1_utils import pop_names, pop_name_to_cell_type

# This must be the same source network used to generate the recording.
data_dir = Path(metadata["data_dir"])
if not data_dir.is_absolute():
    data_dir = repo_root / data_dir

population_names = pop_names(
    {"tf_id_to_bmtk_id": bmtk_ids}, data_dir=str(data_dir)
)
cell_types = np.array([
    pop_name_to_cell_type(name, ignore_l5e_subtypes=False)
    for name in population_names
])
assert len(cell_types) == len(neuron_ids) == len(bmtk_ids)

neuron_info = pd.DataFrame({
    "column": np.arange(len(neuron_ids)),
    "neuron_id": neuron_ids,
    "bmtk_id": bmtk_ids,
    "population": population_names,
    "cell_type": cell_types,
})
neuron_info.head(10)


I0000 00:00:1793185661.048045 2473737 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793185661.113916 2473737 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793185663.327476 2473737 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


,column,neuron_id,bmtk_id,population,cell_type
0,0,5,5,i1Htr3a,L1 Htr3a
1,1,13,13,i1Htr3a,L1 Htr3a
2,2,18,18,i1Htr3a,L1 Htr3a
3,3,20,20,i1Htr3a,L1 Htr3a
4,4,24,24,i1Htr3a,L1 Htr3a
5,5,25,25,i1Htr3a,L1 Htr3a
6,6,31,31,i1Htr3a,L1 Htr3a
7,7,33,33,i1Htr3a,L1 Htr3a
8,8,34,34,i1Htr3a,L1 Htr3a
9,9,36,36,i1Htr3a,L1 Htr3a


In [8]:
# cell_types[i] describes column i of both full recording datasets.
# The loaded windows contain only their first n_columns columns.
window_cell_types = cell_types[:n_columns]
l4_exc_spikes = spikes[:, window_cell_types == "L4 Exc"]
print(neuron_info["cell_type"].value_counts())
print(f"Loaded L4 Exc spikes: {l4_exc_spikes.shape}")


cell_type
L4 Exc      17390
L2/3 Exc    16755
L6 Exc      15028
L5 IT        6534
L5 ET        2349
L4 PV        1155
L5 PV        1103
L5 SST        954
L2/3 VIP      923
L4 SST        640
L1 Htr3a      601
L2/3 PV       560
L6 SST        553
L6 PV         506
L2/3 SST      387
L5 NP         373
L4 VIP        323
L5 VIP        185
L6 VIP         89
Name: count, dtype: int64
Loaded L4 Exc spikes: (60000, 17390)


## A short view of the recording

The figure shows a **1-second window**, independently of the analysis window above. It reuses loaded data when that window is already in memory. The raster uses `plotting_utils.LaminarPlot`, as in `callbacks.py`: neurons are grouped by layer and population, sorted by tuning angle within each group, and colored with the shared population palette. Alternating shaded bands identify cortical layers. `raster_max_neurons` controls how many recorded neurons are sampled evenly across the columns.

The second panel shows normalized voltages for up to four sampled neurons, identified by canonical neuron ID and cell type. Times are measured from the end of warm-up. The raster needs only source-neuron metadata; no network reconstruction is required.


In [9]:
figure_start_ms = 0
figure_window_ms = 300
raster_max_neurons = max_neurons
voltage_max_neurons = 4


In [10]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
from v1_model_utils.plotting_utils import CELL_CLASS_COLORS, LaminarPlot

if figure_start_ms < 0 or min(figure_window_ms, raster_max_neurons, voltage_max_neurons) <= 0:
    raise ValueError("Use a nonnegative figure start and positive duration/neuron counts.")
figure_first = int(figure_start_ms / dt_ms)
figure_last = min(
    figure_first + max(1, int(figure_window_ms / dt_ms)), completed_steps
)
if figure_first >= figure_last:
    raise ValueError("Choose a figure window within the completed recording.")

# HDF5 reads only this short window and the selected columns.
figure_columns = np.linspace(
    0, len(neuron_ids) - 1, min(raster_max_neurons, len(neuron_ids)), dtype=int
)
trace_columns = figure_columns[np.linspace(
    0, len(figure_columns) - 1, min(voltage_max_neurons, len(figure_columns)), dtype=int
)]
# Reuse the loaded arrays when they cover this figure's times and columns.
figure_rows = slice(figure_first - start_step, figure_last - start_step)
window_covers_figure = start_step <= figure_first and figure_last <= stop_step
all_figure_columns = len(figure_columns) == len(neuron_ids)
figure_selection = slice(None) if all_figure_columns else figure_columns
with h5py.File(recording_path, "r") as recording:
    if window_covers_figure and figure_columns[-1] < n_columns:
        figure_spikes = spikes[figure_rows, figure_selection]
    elif all_figure_columns:
        figure_spikes = read_window(
            recording["spikes"], figure_first, figure_last, len(neuron_ids), load_workers
        )
    else:
        figure_spikes = recording["spikes"][figure_first:figure_last, figure_columns]
    if window_covers_figure and trace_columns[-1] < n_columns:
        figure_voltages = voltages[figure_rows, trace_columns]
    else:
        figure_voltages = recording["voltages"][figure_first:figure_last, trace_columns]

fig, (raster_ax, voltage_ax) = plt.subplots(
    2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)
# Match the callbacks' laminar raster using metadata for these exact columns.
with h5py.File(data_dir / "network/v1_nodes.h5", "r") as nodes:
    tuning_angles = nodes["nodes/v1/0/tuning_angle"][:][bmtk_ids[figure_columns]]
raster_network = {
    "n_nodes": len(figure_columns),
    "tf_id_to_bmtk_id": bmtk_ids[figure_columns],
    "tuning_angle": tuning_angles,
}
# The recording is already core-only, and gray input has no stimulus boundaries.
laminar_raster = LaminarPlot(
    raster_network, str(data_dir), plot_core_only=False,
    stimuli_init_time=-1, stimuli_end_time=-1,
    scale=1.5, marker_size=1, alpha=0.65,
)
laminar_raster(raster_ax, figure_spikes[np.newaxis])
raster_ax.set_title(f"A. Core activity ({len(figure_columns):,} neurons)", loc="left")
raster_ax.set_ylabel("Neuron (layer order)", fontsize=11)
raster_ax.legend(
    loc="upper right", ncol=5, fontsize=8,
    frameon=True, facecolor="white", framealpha=0.95,
)

sample_steps = np.arange(figure_last - figure_first)
population_masks = {
    "Exc": laminar_raster.e_mask,
    "Htr3a": laminar_raster.htr3a_mask,
    "Vip": laminar_raster.vip_mask,
    "Sst": laminar_raster.sst_mask,
    "Pvalb": laminar_raster.pvalb_mask,
}
for trace, column in enumerate(trace_columns):
    raster_column = np.searchsorted(figure_columns, column)
    trace_color = next(
        CELL_CLASS_COLORS[name]
        for name, mask in population_masks.items() if mask[raster_column]
    )
    voltage_ax.plot(
        sample_steps, figure_voltages[:, trace].astype(np.float32),
        color=trace_color, linewidth=0.9,
        label=f"Neuron {neuron_ids[column]} ({cell_types[column]})",
    )
voltage_ax.axhline(0, color="0.6", linewidth=0.7, linestyle=":")
voltage_ax.axhline(1, color="0.6", linewidth=0.7, linestyle=":")
voltage_ax.set_title("B. Membrane voltage", loc="left")
voltage_ax.set_ylabel("Normalized voltage")
voltage_ax.set_xlabel("Time after warm-up (ms)")
voltage_ax.legend(fontsize=8, loc="upper right")
    voltage_ax.set_xlim(0, max(1, len(sample_steps) - 1))
voltage_ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
voltage_ax.xaxis.set_major_formatter(FuncFormatter(
    lambda step, _: f"{(figure_first + step + 1) * dt_ms:g}"
))
for ax in (raster_ax, voltage_ax):
    ax.tick_params(axis="both", labelsize=10)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


IndentationError: unexpected indent (1637600148.py, line 90)